# 📊 Notebook 1 — Exploratory Data Analysis
## Aerial Bird vs Drone Classification

**Objectives:**
- Understand dataset structure
- Count images per class & split
- Detect class imbalance
- Visualize sample images
- Analyze image size distribution
- Compute pixel statistics (mean, std)

In [ ]:
# ── Setup ─────────────────────────────────────────────────────
import os, sys
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from collections import Counter
from pathlib import Path

# Add project root to path
ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(ROOT))

from src.config import CLASSIFICATION_DIR, TRAIN_DIR, VALID_DIR, TEST_DIR
from src.utils import (
    plot_sample_images, plot_class_distribution,
    count_images, get_image_stats
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ Imports successful')
print(f'Dataset root: {CLASSIFICATION_DIR}')

## 1. Dataset Structure

In [ ]:
# Count images per split and class
split_counts = {}
for split in ['TRAIN', 'VALID', 'TEST']:
    split_dir = os.path.join(CLASSIFICATION_DIR, split)
    split_counts[split] = count_images(split_dir)

print('\n' + '='*50)
print('  DATASET SUMMARY')
print('='*50)
print(f'{"Split":<10} {"Bird":>8} {"Drone":>8} {"Total":>8}')
print('-'*50)
grand_total = 0
for split, counts in split_counts.items():
    bird_n  = counts.get('bird', 0)
    drone_n = counts.get('drone', 0)
    total   = bird_n + drone_n
    grand_total += total
    print(f'{split:<10} {bird_n:>8} {drone_n:>8} {total:>8}')
print('-'*50)
print(f'{"TOTAL":<10} {"":>8} {"":>8} {grand_total:>8}')
print('='*50)

In [ ]:
# Class Distribution Bar Chart
plot_class_distribution(
    split_counts,
    save_path=str(ROOT / 'results' / 'plots' / 'class_distribution.png')
)

## 2. Sample Images Visualization

In [ ]:
# Display sample training images
plot_sample_images(
    TRAIN_DIR,
    classes=['bird', 'drone'],
    n=6,
    title='Training Set — Sample Images',
    save_path=str(ROOT / 'results' / 'plots' / 'sample_images.png')
)

## 3. Image Size Distribution

In [ ]:
# Collect image dimensions from training set
widths, heights = [], []
for cls in ['bird', 'drone']:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            img = Image.open(os.path.join(cls_dir, fname))
            w, h = img.size
            widths.append(w)
            heights.append(h)

print(f'Total train images analyzed: {len(widths)}')
print(f'Width  — min:{min(widths)}, max:{max(widths)}, mean:{np.mean(widths):.0f}')
print(f'Height — min:{min(heights)}, max:{max(heights)}, mean:{np.mean(heights):.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=25, color='#3F51B5', alpha=0.8, edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Count')

axes[1].hist(heights, bins=25, color='#4CAF50', alpha=0.8, edgecolor='white')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')
axes[1].set_ylabel('Count')

plt.suptitle('Image Size Distribution (Training Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ROOT / 'results' / 'plots' / 'image_size_dist.png'), dpi=150)
plt.show()

## 4. Pixel Statistics

In [ ]:
# Compute per-channel mean and std for normalization reference
print('Computing pixel statistics (this may take a moment)...')
stats = get_image_stats(TRAIN_DIR)

print(f'\nTraining Set Pixel Statistics:')
print(f'  Global Mean : {stats["mean"]:.4f}')
print(f'  Global Std  : {stats["std"]:.4f}')
print(f'  Images      : {stats["n_images"]}')
print('\n→ Normalization: pixel / 255.0 (scale to [0, 1])')

## 5. Class Imbalance Analysis

In [ ]:
# Compute imbalance ratio
train_bird  = split_counts['TRAIN'].get('bird', 0)
train_drone = split_counts['TRAIN'].get('drone', 0)

ratio = max(train_bird, train_drone) / (min(train_bird, train_drone) + 1e-8)
minority = 'bird' if train_bird < train_drone else 'drone'

print(f'Training set:  bird={train_bird}, drone={train_drone}')
print(f'Imbalance ratio: {ratio:.2f}x')

if ratio > 2.0:
    print(f'\n⚠️  Class imbalance detected (ratio > 2x).')
    print(f'   Minority class: {minority}')
    print('   Mitigation strategies:')
    print('     1. Data augmentation (already applied)')
    print('     2. Class weighting in loss function')
    print('     3. Oversampling minority class')
else:
    print('\n✅ Class balance is acceptable (ratio < 2x).')

# Pie chart
fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(
    [train_bird, train_drone],
    labels=['Bird', 'Drone'],
    colors=['#4CAF50', '#f44336'],
    autopct='%1.1f%%',
    startangle=90,
    explode=(0.05, 0.05),
    shadow=True
)
ax.set_title('Training Set — Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ROOT / 'results' / 'plots' / 'class_pie.png'), dpi=150)
plt.show()

## 6. Summary & EDA Conclusions

In [ ]:
print('='*60)
print('  EDA CONCLUSIONS')
print('='*60)
print(f'  Dataset: {grand_total} total images across 3 splits')
print(f'  Classes: Bird | Drone (binary classification)')
print(f'  Imbalance Ratio: {ratio:.2f}x → {"apply class weights" if ratio>2 else "acceptable"}')
print(f'  Image sizes vary → resize to 224×224 for uniform input')
print(f'  Pixel range [0,255] → normalize to [0,1]')
print(f'  Augmentation needed: rotation, flip, zoom, brightness')
print('='*60)